In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, precision_recall_curve, roc_curve,
    auc, classification_report, confusion_matrix
)
import numpy as np

In [2]:
df = pd.read_csv('processed_medical_data.csv')

#separate
X = df.drop('No-show', axis=1).values.astype(np.float32)
y = df['No-show'].values.astype(np.int64) #somehow fixes with long

In [3]:
class NoShowDataset(Dataset):
    def __init__(self, features, labels):
        self.x = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [4]:
#Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify = y
)

In [5]:
# Create dataset and dataloaders
train_dataset = NoShowDataset(X_train, y_train)
test_dataset = NoShowDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [6]:
class NN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(NN, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)           
        self.fc2 = nn.Linear(64, 32)           
        self.out = nn.Linear(32, num_classes)  

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)
        return x

In [7]:
#Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [8]:
# Model parameters
input_size = X.shape[1]
num_classes = 2
num_epochs=100

In [9]:
# Initialize model, loss function and optimizer
class_weights = [1.0, 1.5]  # [class_0_weight, class_1_weight]
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

model = NN(input_size=input_size, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

initial_lr = 0.03
optimizer = optim.Adam(model.parameters(), lr=initial_lr, betas=(0.9, 0.999), weight_decay=5e-4)

# Inbuilt exponential decay
gamma = 0.95  # Learning rate decays by 5% each epoch
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

print(f'Model initialized with input size: {input_size}')
print(f'Training samples: {len(train_dataset)}, Test samples: {len(test_dataset)}')

Model initialized with input size: 343
Training samples: 88421, Test samples: 22106


In [10]:
#Training function
def train_model():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device)
        targets = targets.to(device)

        #Forward pass
        outputs = model(data)
        loss = criterion(outputs, targets)

        #Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        #Statistics
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        # if batch_idx % 20 == 0:
        #     print(f'Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}, LR: {optimizer.param_groups[0]["lr"]:.6f}')

    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

In [11]:
def train_single_epoch(epoch):
    # Train the model
    train_loss, train_acc = train_model()

    # Step the scheduler to apply exponential decay
    scheduler.step()
    
    # Get current learning rate from optimizer
    current_lr = optimizer.param_groups[0]['lr']
    
    return train_loss, train_acc, current_lr

In [12]:
# Testing function with custom threshold
def test_model(threshold=0.65):
    model.eval()
    test_loss = 0
    all_predictions = []
    all_targets = []
    all_probabilities = []

    with torch.no_grad():
        for data, targets in test_loader:
            data = data.to(device)
            targets = targets.to(device)

            outputs = model(data)
            loss = criterion(outputs, targets)

            test_loss += loss.item()

            # Get probabilities
            probabilities = F.softmax(outputs, dim=1)
            # Apply custom threshold o
            preds = (probabilities[:, 1] > threshold).long()

            all_predictions.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

        # Convert lists to numpy arrays
        all_predictions = np.array(all_predictions)
        all_targets = np.array(all_targets)
        all_probabilities = np.array(all_probabilities)

        # Calculate metrics
        accuracy = accuracy_score(all_targets, all_predictions)

        # F1 scores for both classes
        f1_macro = f1_score(all_targets, all_predictions, average='macro')
        f1_per_class = f1_score(all_targets, all_predictions, average=None)
        f1_no_show = f1_per_class[1]  
        f1_show = f1_per_class[0]     

        # ROC AUC (uses continuous probabilities for class 1)
        fpr, tpr, _ = roc_curve(all_targets, all_probabilities[:, 1])
        roc_auc = auc(fpr, tpr)

        # PR AUC (also uses probabilities for class 1)
        precision_vals, recall_vals, _ = precision_recall_curve(all_targets, all_probabilities[:, 1])
        pr_auc = auc(recall_vals, precision_vals)

        avg_loss = test_loss / len(test_loader)

        return {
            'loss': avg_loss,
            'accuracy': accuracy * 100,
            'f1_macro': f1_macro,
            'f1_no_show': f1_no_show,
            'f1_show': f1_show,
            'roc_auc': roc_auc,
            'pr_auc': pr_auc,
            'predictions': all_predictions,
            'targets': all_targets,
            'probabilities': all_probabilities
        }

In [13]:
# Training loop
training_history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}
for epoch in range(num_epochs+1):
    train_loss, train_acc, current_lr = train_single_epoch(epoch)
    test_metrics = test_model(0.65)
    
    # Store metrics
    training_history['train_loss'].append(train_loss)
    training_history['train_acc'].append(train_acc)
    training_history['val_loss'].append(test_metrics['loss'])
    training_history['val_acc'].append(test_metrics['accuracy'])
    
    if epoch % 5 == 0:
        print(f'Epoch [{epoch}/{num_epochs}]')
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, LR: {current_lr:.6f}')
        print(f'Val Loss: {test_metrics["loss"]:.4f}, Val Acc: {test_metrics["accuracy"]:.2f}%')

Epoch [0/100]
Train Loss: 0.3090, Train Acc: 87.80%, LR: 0.028500
Val Loss: 0.2073, Val Acc: 91.03%
Epoch [5/100]
Train Loss: 0.2017, Train Acc: 90.83%, LR: 0.022053
Val Loss: 0.2005, Val Acc: 91.10%
Epoch [10/100]
Train Loss: 0.1977, Train Acc: 90.88%, LR: 0.017064
Val Loss: 0.1993, Val Acc: 90.56%
Epoch [15/100]
Train Loss: 0.1950, Train Acc: 91.02%, LR: 0.013204
Val Loss: 0.1943, Val Acc: 90.86%
Epoch [20/100]
Train Loss: 0.1917, Train Acc: 91.09%, LR: 0.010217
Val Loss: 0.2026, Val Acc: 91.30%
Epoch [25/100]
Train Loss: 0.1908, Train Acc: 91.08%, LR: 0.007906
Val Loss: 0.1895, Val Acc: 91.78%
Epoch [30/100]
Train Loss: 0.1887, Train Acc: 91.16%, LR: 0.006117
Val Loss: 0.1881, Val Acc: 91.17%
Epoch [35/100]
Train Loss: 0.1874, Train Acc: 91.23%, LR: 0.004733
Val Loss: 0.1911, Val Acc: 91.61%
Epoch [40/100]
Train Loss: 0.1855, Train Acc: 91.25%, LR: 0.003663
Val Loss: 0.1874, Val Acc: 91.44%
Epoch [45/100]
Train Loss: 0.1845, Train Acc: 91.30%, LR: 0.002834
Val Loss: 0.1916, Val Acc:

In [18]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve

def test_model():
    model.eval()
    preds, targets, probs = [], [], []
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            out = model(data)
            prob = F.softmax(out, dim=1)
            _, pred = out.max(1)
            preds.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())
            probs.extend(prob.cpu().numpy())
    
    preds = np.array(preds)
    targets = np.array(targets)
    probs = np.array(probs)
    
    accuracy = accuracy_score(targets, preds) * 100
    f1_macro = f1_score(targets, preds, average='macro')
    f1_per_class = f1_score(targets, preds, average=None)
    fpr, tpr, _ = roc_curve(targets, probs[:,1])
    roc_auc = auc(fpr, tpr)
    precision, recall, _ = precision_recall_curve(targets, probs[:,1])
    pr_auc = auc(recall, precision)
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_no_show': f1_per_class[1],
        'f1_show': f1_per_class[0],
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'predictions': preds,
        'targets': targets,
        'probabilities': probs
    }

metrics = test_model()

print("Classification Report:")
print(classification_report(metrics['targets'], metrics['predictions'], target_names=['No Show', 'Show'], digits=4))

cm = confusion_matrix(metrics['targets'], metrics['predictions'])
print("Confusion matrix: ")
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"\nBreakdown:")
print(f"True Negatives (Showed up correctly): {tn}")
print(f"False Positives (Predicted no-show, but showed up): {fp}")
print(f"False Negatives (Predicted showed up, but no-show): {fn}")
print(f"True Positives (No-show correctly predicted): {tp}")
print(f"\nSummary:")
print(f"Accuracy: {metrics['accuracy']:.2f}%")
print(f"F1 Macro: {metrics['f1_macro']:.4f}")
print(f"F1 No-Show: {metrics['f1_no_show']:.4f}")
print(f"F1 Show: {metrics['f1_show']:.4f}")
print(f"ROC AUC: {metrics['roc_auc']:.4f}")
print(f"PR AUC: {metrics['pr_auc']:.4f}")

no_show_acc = accuracy_score(metrics['targets'][metrics['targets'] == 0], metrics['predictions'][metrics['targets'] == 0]) * 100
show_acc = accuracy_score(metrics['targets'][metrics['targets'] == 1], metrics['predictions'][metrics['targets'] == 1]) * 100
print(f"\nPer-class Accuracy:")
print(f"No-Show: {no_show_acc:.2f}%")
print(f"Show: {show_acc:.2f}%")

Classification Report:
              precision    recall  f1-score   support

     No Show     0.9591    0.9289    0.9438     17642
        Show     0.7502    0.8436    0.7942      4464

    accuracy                         0.9117     22106
   macro avg     0.8547    0.8863    0.8690     22106
weighted avg     0.9170    0.9117    0.9136     22106

Confusion matrix: 
[[16388  1254]
 [  698  3766]]

Breakdown:
True Negatives (Showed up correctly): 16388
False Positives (Predicted no-show, but showed up): 1254
False Negatives (Predicted showed up, but no-show): 698
True Positives (No-show correctly predicted): 3766

Summary:
Accuracy: 91.17%
F1 Macro: 0.8690
F1 No-Show: 0.7942
F1 Show: 0.9438
ROC AUC: 0.9704
PR AUC: 0.8998

Per-class Accuracy:
No-Show: 92.89%
Show: 84.36%
